# Boid Simulation Exploration

Basic exploration of the Boid flocking simulation.
This notebook demonstrates:
- Running the simulation
- Visualizing agent behavior
- Analyzing the interaction network

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from src.models.boid_model import BoidModel

## 1. Run Simulation

In [ ]:
model = BoidModel(n_agents=50, width=200, height=200, perception_radius=50)

# Run 100 steps
for i in range(100):
    model.step()

print(f"Steps completed: {model.step_count}")
print(f"Interaction edges: {model.network.number_of_edges()}")

## 2. Agent Positions

In [ ]:
positions = model.get_agent_positions()
velocities = model.get_agent_velocities()

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(positions[:, 0], positions[:, 1], s=20, alpha=0.7, c='steelblue')
ax.quiver(positions[:, 0], positions[:, 1], velocities[:, 0], velocities[:, 1],
          alpha=0.5, scale=30, width=0.003)
ax.set_xlim(0, model.space.width)
ax.set_ylim(0, model.space.height)
ax.set_aspect('equal')
ax.set_title(f'Boid Positions at Step {model.step_count}')
plt.tight_layout()
plt.show()

## 3. Interaction Network

In [ ]:
G = model.get_interaction_network()
fig, ax = plt.subplots(figsize=(8, 8))
pos_layout = nx.spring_layout(G, seed=42)
degrees = dict(G.degree())
node_sizes = [v * 20 + 30 for v in degrees.values()]
nx.draw(G, pos=pos_layout, ax=ax, node_size=node_sizes,
        node_color='steelblue', edge_color='gray', alpha=0.7, width=0.5)
ax.set_title(f'Interaction Network ({G.number_of_edges()} edges)')
plt.tight_layout()
plt.show()

## 4. Network Statistics

In [ ]:
from src.utils.network import network_stats
stats = network_stats(G)
for key, value in stats.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

## 5. Data Collection

In [ ]:
df = model.datacollector.get_model_vars_dataframe()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df['n_edges'].plot(ax=axes[0, 0], title='Interaction Edges')
df['avg_degree'].plot(ax=axes[0, 1], title='Average Degree')
df['n_components'].plot(ax=axes[1, 0], title='Connected Components')
df['avg_clustering'].plot(ax=axes[1, 1], title='Average Clustering')
plt.tight_layout()
plt.show()